# TurboQuant VRAM Test

Run on Colab with a GPU runtime (A100/L4 recommended).
Compares baseline vs TurboQuant VRAM usage with the fused Triton attention kernel.

**How to read results:** TurboQuant compresses the KV cache, not the model
weights. At 8K tokens the KV cache is ~460 MB for Qwen 2.5 7B — TurboQuant
cuts that by ~69%. For dramatic total-VRAM savings, use longer context
(32K+) where the KV cache dominates.

In [ ]:
!pip install -q git+https://github.com/Echen1246/local-turboquant.git@triton-fused-attention

In [ ]:
import torch
import gc
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")
try:
    import triton
    print(f"Triton version: {triton.__version__}")
except ImportError:
    print("Triton not installed — will use PyTorch fallback attention")

In [ ]:
from turboquant import TurboQuantSession

MODEL = "Qwen/Qwen2.5-7B-Instruct"
DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"

PROMPT = "The quick brown fox " * 2000  # ~8K tokens
BITS = 4
MAX_NEW = 32

print(f"Model: {MODEL}")
print(f"dtype: {DTYPE}")
print(f"VRAM before loading: {torch.cuda.memory_allocated() / 1e9:.3f} GB")

In [ ]:
# --- Baseline ---
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

pre_bl = torch.cuda.memory_allocated()
print(f"VRAM before baseline load: {pre_bl / 1e9:.3f} GB")

session_bl = TurboQuantSession.from_pretrained(
    MODEL, variant="baseline", bits=BITS,
    dtype=DTYPE, device_map="auto",
)
model_bytes_bl = torch.cuda.memory_allocated()
print(f"Model loaded: {model_bytes_bl / 1e9:.2f} GB  (model weights ≈ {(model_bytes_bl - pre_bl) / 1e9:.2f} GB)")

# Sanity: Qwen 2.5 7B in BF16 should be ~15 GB, FP32 ~30 GB
param_bytes = sum(p.numel() * p.element_size() for p in session_bl.model.parameters())
print(f"Actual param footprint: {param_bytes / 1e9:.2f} GB  (dtype={next(session_bl.model.parameters()).dtype})")

text_bl = session_bl.generate(prompt=PROMPT, max_new_tokens=MAX_NEW)
peak_bl = torch.cuda.max_memory_allocated()
telem_bl = session_bl.last_telemetry()

print(f"\nBaseline peak VRAM: {peak_bl / 1e9:.2f} GB")
print(f"Baseline gen time: {telem_bl.get('generation_seconds', 'N/A')}s")
print(f"Output: {text_bl[:200]}...")

del session_bl, text_bl
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# --- TurboQuant ---
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

pre_tq = torch.cuda.memory_allocated()
print(f"VRAM before TQ load: {pre_tq / 1e9:.3f} GB  (residual from baseline cleanup)")

session_tq = TurboQuantSession.from_pretrained(
    MODEL, variant="qmse_packed", bits=BITS,
    dtype=DTYPE, device_map="auto",
)
model_bytes_tq = torch.cuda.memory_allocated()
print(f"Model loaded: {model_bytes_tq / 1e9:.2f} GB  (model weights ≈ {(model_bytes_tq - pre_tq) / 1e9:.2f} GB)")

param_bytes_tq = sum(p.numel() * p.element_size() for p in session_tq.model.parameters())
print(f"Actual param footprint: {param_bytes_tq / 1e9:.2f} GB  (dtype={next(session_tq.model.parameters()).dtype})")

text_tq = session_tq.generate(prompt=PROMPT, max_new_tokens=MAX_NEW)
peak_tq = torch.cuda.max_memory_allocated()
telem_tq = session_tq.last_telemetry()

print(f"\nTurboQuant peak VRAM: {peak_tq / 1e9:.2f} GB")
print(f"TurboQuant gen time: {telem_tq.get('generation_seconds', 'N/A')}s")
print(f"Packed KV: {telem_tq.get('packed_actual_bytes', 0) / 1e6:.0f} MB")
print(f"Dense KV would be: {telem_tq.get('dense_kv_bytes', 0) / 1e6:.0f} MB")
print(f"Payload savings: {telem_tq.get('payload_savings_percent', 0):.1f}%")

In [ ]:
# --- Comparison ---
dense_kv_mb = telem_tq.get('dense_kv_bytes', 0) / 1e6
packed_kv_mb = telem_tq.get('packed_actual_bytes', 0) / 1e6
payload_pct = telem_tq.get('payload_savings_percent', 0)
kv_saved_mb = dense_kv_mb - packed_kv_mb

print("=" * 70)
print(f"{'Metric':<35} {'Baseline':>16} {'TurboQuant':>16}")
print("-" * 70)
print(f"{'Peak VRAM (GB)':<35} {peak_bl/1e9:>16.2f} {peak_tq/1e9:>16.2f}")
print(f"{'Dense KV cache (MB)':<35} {dense_kv_mb:>16.0f} {dense_kv_mb:>16.0f}")
print(f"{'Packed KV cache (MB)':<35} {'—':>16} {packed_kv_mb:>16.0f}")
print(f"{'KV cache saved (MB)':<35} {'—':>16} {kv_saved_mb:>16.0f}")
print(f"{'KV payload savings':<35} {'—':>16} {payload_pct:>15.1f}%")
print(f"{'Gen time (s)':<35} {telem_bl.get('generation_seconds','?'):>16} {telem_tq.get('generation_seconds','?'):>16}")
print("=" * 70)

print(f"\nModel param dtype (baseline): {next(iter([])) if False else 'see cell 4 output'}")
print(f"Note: both runs load the SAME model. VRAM difference = KV cache savings.")
print(f"      At 8K tokens, KV cache is only {dense_kv_mb:.0f} MB vs {param_bytes / 1e9:.1f} GB model weights.")
print(f"      For large savings, use 32K+ context where KV cache is several GB.")